## Preliminares

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer
import sklearn.impute as skl_imp
from sklearn.experimental import enable_iterative_imputer # Necesario para usar skl_imp, no borrar
from src.config import data_folder
from src.transform import *

In [2]:
# Abrir archivo raw_data
df = pd.read_parquet(f"{data_folder}/raw_data.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11391 entries, 0 to 11390
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   Date                         11391 non-null  datetime64[ns]
 1   Ticker                       11391 non-null  object        
 2   Close                        11391 non-null  float64       
 3   Open                         11391 non-null  float64       
 4   Volume                       11391 non-null  float64       
 5   DateAdded                    6705 non-null   object        
 6   Sector                       11391 non-null  object        
 7   Industry                     11391 non-null  object        
 8   TotalRevenue                 11391 non-null  float64       
 9   GrossProfit                  10523 non-null  float64       
 10  OperatingIncome              11391 non-null  float64       
 11  NetIncome                    11391 non-nu

* Para mejorar la visualización de los datos, se expresan las columnas financieras y volumen en miles:

In [3]:
df = financieras_en_millones(df)
df['Volume'] = df['Volume']/10**6

In [4]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Date,11391,2023-08-23 16:59:09.749802240,2020-09-01 00:00:00,2022-03-01 00:00:00,2023-09-01 00:00:00,2025-03-01 00:00:00,2026-06-01 00:00:00,NaN
Close,11391.0,1038.957576,0.57,41.111301,82.443237,164.889503,775000.0,22846.03821
Open,11391.0,1006.586258,0.53,40.56363,81.300003,161.215086,775648.0,22096.142694
Volume,11391.0,328.636592,0.0002,44.14905,103.2401,257.9517,36280.458,1250.868956
TotalRevenue,11391.0,6587.181908,-12134.0,1277.1005,2389.0,5259.0,227173.0,14888.047393
GrossProfit,10523.0,2361.543956,-24353.0,422.542,832.2,1828.0,137192.0,6374.194846
OperatingIncome,11391.0,966.680233,-55512.0,125.685,304.0,820.0,77648.0,2942.496474
NetIncome,11391.0,717.643039,-43621.0,65.5045,199.2,580.0,62578.0,2669.958899
EBITDA,11391.0,1143.504687,-55512.0,155.0,370.333,978.0,84427.0,3366.033896
BasicAverageShares,11345.0,641.186144,0.00021,95.479927,219.67,550.277,30864.0,1742.340602


## Corrección de errores

In [26]:
df['DepreciationAndAmortization'].describe()

count    11324.000000
mean       -38.990404
std        687.871200
min     -29774.000000
25%        -47.673000
50%          0.000000
75%          0.000000
max      19471.000000
Name: DepreciationAndAmortization, dtype: float64

* Se detectan problemas de *signo mixto*. La gran mayoría de los valores se ingresaron con signo negativo, y existen casos donde se cargaron con el signo positivo, la cual es la forma correcta de expresarlo.

In [24]:
df[df['DepreciationAndAmortization']>0]['Ticker'].count()

np.int64(1840)

In [25]:
df[df['DepreciationAndAmortization']<0]['Ticker'].count()

np.int64(4284)

In [5]:
# Definir las columnas a visualizar
columnas_clave = ['Ticker', 'Date', 'TotalDebt', 'CurrentDebt', 'LongTermDebt']

# Usar .loc[condicion_de_filas, lista_de_columnas]
anomalias = df.loc[df['TotalDebt'] < 0, columnas_clave] # No pueden existir valores negativos de deuda

print(anomalias)

      Ticker       Date  TotalDebt  CurrentDebt  LongTermDebt
9468    STLD 2024-03-01  -2826.550      459.987     -3286.537
9469    STLD 2024-06-01  -3144.332      425.696     -3570.028
9471    STLD 2024-12-01  -3115.335      882.013     -3997.348
10252    TXN 2025-03-01 -27299.000      750.000    -28049.000


Se observa en los 4 casos que:
- TotalDebt es negativa: anormal
- CurrentDebt es positiva: correcto
- LongTermDebt es negativa: anormal

Al hacer la suma, se cumple la igualdad `TotalDebt` = `CurrentDebt` + `LongTermDebt`.
Esto indica que se trato de un error al parsear los datos.

Se consideran entonces errores de signo.

In [6]:
# Identificar las filas donde LongTermDebt es negativo
filtro_deuda = df['LongTermDebt'] < 0

# Se corrigen usando valor absoluto
df.loc[filtro_deuda, 'LongTermDebt'] = df['LongTermDebt'].abs()

# Recalcular la Deuda Total
df.loc[filtro_deuda, 'TotalDebt'] = df['CurrentDebt'] + df['LongTermDebt']

# Verificar 
anomalias = df.loc[df['TotalDebt'] < 0, columnas_clave]
print(anomalias)

Empty DataFrame
Columns: [Ticker, Date, TotalDebt, CurrentDebt, LongTermDebt]
Index: []


* Negativos en `CurrentDebt`:

In [7]:
anomalias = df.loc[df['CurrentDebt'] < 0, columnas_clave]
print(anomalias)

     Ticker       Date  TotalDebt  CurrentDebt  LongTermDebt
5158    IEP 2022-09-01     6388.0       -746.0        7134.0
5159    IEP 2022-12-01     6382.0       -745.0        7127.0


Aquí se observa el mismo comportamiento, pero con errores de signo en `CurrentDebt`.

In [8]:
# Se corrigen los valores
filtro_deuda = df['CurrentDebt'] < 0
df.loc[filtro_deuda, 'CurrentDebt'] = df['CurrentDebt'].abs()
df.loc[filtro_deuda, 'TotalDebt'] = df['CurrentDebt'] + df['LongTermDebt']

# Verificar
anomalias = df.loc[df['CurrentDebt'] < 0, columnas_clave]
print(anomalias)

Empty DataFrame
Columns: [Ticker, Date, TotalDebt, CurrentDebt, LongTermDebt]
Index: []


In [9]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Date,11391,2023-08-23 16:59:09.749802240,2020-09-01 00:00:00,2022-03-01 00:00:00,2023-09-01 00:00:00,2025-03-01 00:00:00,2026-06-01 00:00:00,NaN
Close,11391.0,1038.957576,0.57,41.111301,82.443237,164.889503,775000.0,22846.03821
Open,11391.0,1006.586258,0.53,40.56363,81.300003,161.215086,775648.0,22096.142694
Volume,11391.0,328.636592,0.0002,44.14905,103.2401,257.9517,36280.458,1250.868956
TotalRevenue,11391.0,6587.181908,-12134.0,1277.1005,2389.0,5259.0,227173.0,14888.047393
GrossProfit,10523.0,2361.543956,-24353.0,422.542,832.2,1828.0,137192.0,6374.194846
OperatingIncome,11391.0,966.680233,-55512.0,125.685,304.0,820.0,77648.0,2942.496474
NetIncome,11391.0,717.643039,-43621.0,65.5045,199.2,580.0,62578.0,2669.958899
EBITDA,11391.0,1143.504687,-55512.0,155.0,370.333,978.0,84427.0,3366.033896
BasicAverageShares,11345.0,641.186144,0.00021,95.479927,219.67,550.277,30864.0,1742.340602


* Se analizan a continuación anomalías en `TotalRevenue`, los ingresos no pueden ser negativos:

In [10]:
columnas_clave = ['Ticker', 'Date', 'TotalRevenue', 'OperatingIncome']

anomalias = df.loc[df['TotalRevenue'] < 0, columnas_clave]
print(anomalias)

      Ticker       Date  TotalRevenue  OperatingIncome
414      ALB 2021-03-01   -1790.31392      -114.758843
955     AVNT 2021-03-01     -30.20000       -12.700000
1182     BAX 2023-03-01    -704.00000      -285.000000
1286     BHF 2022-03-01    -150.00000      2266.000000
1290     BHF 2023-03-01    -127.00000        40.000000
2279     CNA 2023-03-01   -3542.00000     -4648.000000
2920     DAN 2024-03-01    -448.00000      -387.000000
3159    DLTR 2024-06-01   -5182.80000       765.300000
3550     EMR 2021-12-01    -357.00000      -206.000000
4115    FTNT 2022-03-01   -1415.00000       213.700000
4145     FTV 2024-03-01    -567.70000      -248.800000
4236      GE 2023-03-01  -12134.00000      4010.000000
4857     HLT 2024-03-01   -3218.00000       438.000000
5504       J 2022-12-01   -1258.70300       -69.426000
5508       J 2023-12-01   -1212.28200      -120.924000
6424     MAR 2024-03-01  -11318.00000       726.000000
6428     MAR 2025-03-01  -12053.00000       804.000000
7209    ND

Se detectan casos imposibles, con ingresos por ventas negativos y beneficios positivos. Pero al no disponer de información de costos para verificar, resulta riesgoso asumir que se trata de errores de signo.

* Por esto se decide imputarlos todos a NaN:

In [11]:
filtro_ingresos = df['TotalRevenue'] < 0
df.loc[filtro_ingresos, 'TotalRevenue'] = np.nan

# Verificar
anomalias = df.loc[df['TotalRevenue'] < 0, columnas_clave]
print(anomalias)

Empty DataFrame
Columns: [Ticker, Date, TotalRevenue, OperatingIncome]
Index: []


* Se imputan parte de los NaNs en variables de Deuda antes de calcular métricas, 
mediante las relaciones contables entre ellas:

In [12]:
df_debt_imputed = imputar_deuda(df)
mostrar_missings(df_debt_imputed)

DateAdded                      0.411377
TotalDebt                      0.208059
CurrentDebt                    0.206391
GrossProfit                    0.076201
LongTermDebt                   0.053814
CurrentAssets                  0.048020
CurrentLiabilities             0.047757
CapitalExpenditure             0.026863
FreeCashFlow                   0.025283
DepreciationAndAmortization    0.005882
BasicAverageShares             0.004038
CashAndCashEquivalents         0.003687
FinancingCashFlow              0.003248
OperatingCashFlow              0.002634
InvestingCashFlow              0.002195
TotalRevenue                   0.002195
StockholdersEquity             0.001405
TotalAssets                    0.001141
NetIncome                      0.000000
Open                           0.000000
Date                           0.000000
OperatingIncome                0.000000
Sector                         0.000000
Industry                       0.000000
Volume                         0.000000


* Se crea feature `YearsSinceAdded`:

In [13]:
#  Pasar DateAdded a formato datetime, los NaN se vuelven NaT (not a time)
df_debt_imputed['DateAdded'] = pd.to_datetime(df_debt_imputed['DateAdded'], errors='coerce')
# Convertir a YearsSinceAdded, aqui los nulos regresan a NaN
df_debt_imputed['YearsSinceAdded'] = round(((pd.Timestamp.now() - df_debt_imputed['DateAdded']).dt.days / 365.25), 0)

# 3. Se asignan a cero años los tickers que no pertenecen al Índice S&P 500
df_debt_imputed['YearsSinceAdded'] = df_debt_imputed['YearsSinceAdded'].fillna(0)

# Eliminar la columna original
df_debt_imputed.drop('DateAdded', axis=1, inplace=True)
mostrar_missings(df_debt_imputed)

TotalDebt                      0.208059
CurrentDebt                    0.206391
GrossProfit                    0.076201
LongTermDebt                   0.053814
CurrentAssets                  0.048020
CurrentLiabilities             0.047757
CapitalExpenditure             0.026863
FreeCashFlow                   0.025283
DepreciationAndAmortization    0.005882
BasicAverageShares             0.004038
CashAndCashEquivalents         0.003687
FinancingCashFlow              0.003248
OperatingCashFlow              0.002634
InvestingCashFlow              0.002195
TotalRevenue                   0.002195
StockholdersEquity             0.001405
TotalAssets                    0.001141
NetIncome                      0.000000
EBITDA                         0.000000
Open                           0.000000
Ticker                         0.000000
Date                           0.000000
OperatingIncome                0.000000
Industry                       0.000000
Volume                         0.000000


* Se aplica forward fill para cubrir posibles huecos de hasta 1 período:

In [14]:
df_debt_imputed = cubrir_huecos(df_debt_imputed,limite=1)
mostrar_missings(df_debt_imputed)

TotalDebt                      0.186551
CurrentDebt                    0.184005
GrossProfit                    0.071723
LongTermDebt                   0.050478
CurrentAssets                  0.047406
CurrentLiabilities             0.047230
CapitalExpenditure             0.022123
FreeCashFlow                   0.021245
DepreciationAndAmortization    0.004126
CashAndCashEquivalents         0.002546
OperatingCashFlow              0.001053
FinancingCashFlow              0.001053
StockholdersEquity             0.000790
InvestingCashFlow              0.000702
BasicAverageShares             0.000702
TotalAssets                    0.000615
EBITDA                         0.000000
NetIncome                      0.000000
OperatingIncome                0.000000
Open                           0.000000
Ticker                         0.000000
Close                          0.000000
Date                           0.000000
TotalRevenue                   0.000000
Sector                         0.000000


* Se convierten las variables de flujo trimestrales a valores TTM (ventana móvil de 4 trimestres):

In [15]:
df_debt_imputed = transformar_flujos_a_ttm(df_debt_imputed)
mostrar_missings(df_debt_imputed)

GrossProfit_TTM                    0.207708
TotalDebt                          0.186551
CurrentDebt                        0.184005
CapitalExpenditure_TTM             0.160653
FreeCashFlow_TTM                   0.159687
DepreciationAndAmortization_TTM    0.140374
FinancingCashFlow_TTM              0.138004
OperatingCashFlow_TTM              0.137828
InvestingCashFlow_TTM              0.137301
BasicAverageShares_TTM             0.136862
TotalRevenue_TTM                   0.136160
OperatingIncome_TTM                0.136160
NetIncome_TTM                      0.136160
EBITDA_TTM                         0.136160
LongTermDebt                       0.050478
CurrentAssets                      0.047406
CurrentLiabilities                 0.047230
CashAndCashEquivalents             0.002546
StockholdersEquity                 0.000790
TotalAssets                        0.000615
Sector                             0.000000
Volume                             0.000000
Industry                        

* Calcular métricas:

In [16]:
df_with_metrics, crecimiento_cols = calcular_metricas(df_debt_imputed)
mostrar_missings(df_with_metrics)

CapEx_YoY                          0.338776
Fcf_YoY                            0.337635
Ebitda_YoY                         0.316566
Revenue_YoY                        0.316566
GrossProfit_TTM                    0.207708
CapEx_QoQ                          0.206303
Fcf_QoQ                            0.204723
PriceToBook                        0.194364
ReturnOnEquity                     0.193749
TotalDebt                          0.186551
CurrentDebt                        0.184005
Revenue_QoQ                        0.181459
Ebitda_QoQ                         0.181459
CapExToRevenue                     0.160653
CapitalExpenditure_TTM             0.160653
FreeCashFlow_TTM                   0.159687
FcfToEbitda                        0.159687
DepreciationAndAmortization_TTM    0.140374
FinancingCashFlow_TTM              0.138004
OperatingCashFlow_TTM              0.137828
NetDebtToEbitda                    0.137828
InvestingCashFlow_TTM              0.137301
MarketCap                       

* Se aplica imputación transversal para las columnas de crecimiento:

In [17]:
df_with_metrics = imputar_transversal(df_with_metrics, crecimiento_cols)
mostrar_missings(df_with_metrics)

GrossProfit_TTM                    0.207708
PriceToBook                        0.194364
ReturnOnEquity                     0.193749
TotalDebt                          0.186551
CurrentDebt                        0.184005
CapExToRevenue                     0.160653
CapitalExpenditure_TTM             0.160653
FreeCashFlow_TTM                   0.159687
FcfToEbitda                        0.159687
DepreciationAndAmortization_TTM    0.140374
FinancingCashFlow_TTM              0.138004
NetDebtToEbitda                    0.137828
OperatingCashFlow_TTM              0.137828
InvestingCashFlow_TTM              0.137301
MarketCap                          0.136862
EnterpriseToEbitda                 0.136862
TrailingPE                         0.136862
EnterpriseValue                    0.136862
BasicAverageShares_TTM             0.136862
OperatingMargins                   0.136160
OperatingIncome_TTM                0.136160
EBITDA_TTM                         0.136160
ReturnOnAssets                  

* Calcular los retornos trimestrales, varianza del activo y covarianza con el mercado para cada ticker:

In [18]:
# Se abre el fichero de precios del Índice del Mercado
df_index = pd.read_parquet(f"{data_folder}/market_index.parquet")
df_with_features = calcular_retornos(df_with_metrics, df_index)
mostrar_missings(df_with_features)

GrossProfit_TTM                    0.207708
PriceToBook                        0.194364
ReturnOnEquity                     0.193749
TotalDebt                          0.186551
CurrentDebt                        0.184005
CapExToRevenue                     0.160653
CapitalExpenditure_TTM             0.160653
FreeCashFlow_TTM                   0.159687
FcfToEbitda                        0.159687
DepreciationAndAmortization_TTM    0.140374
FinancingCashFlow_TTM              0.138004
NetDebtToEbitda                    0.137828
OperatingCashFlow_TTM              0.137828
InvestingCashFlow_TTM              0.137301
EnterpriseValue                    0.136862
BasicAverageShares_TTM             0.136862
EnterpriseToEbitda                 0.136862
MarketCap                          0.136862
TrailingPE                         0.136862
TotalRevenue_TTM                   0.136160
NetIncome_TTM                      0.136160
ReturnOnAssets                     0.136160
ProfitMargins                   

In [19]:
df_with_features.describe().T

,count,mean,min,25%,50%,75%,max,std
Date,11391,2023-08-23 16:59:09.749802240,2020-09-01 00:00:00,2022-03-01 00:00:00,2023-09-01 00:00:00,2025-03-01 00:00:00,2026-06-01 00:00:00,NaN
Close,11391.0,1038.957576,0.57,41.111301,82.443237,164.889503,775000.0,22846.03821
Open,11391.0,1006.586258,0.53,40.56363,81.300003,161.215086,775648.0,22096.142694
Volume,11391.0,328.636592,0.0002,44.14905,103.2401,257.9517,36280.458,1250.868956
CashAndCashEquivalents,11362.0,4522.95146,0.042638,336.0,941.7255,2890.35225,546164.0,19340.379447
CurrentDebt,9295.0,4129.281071,0.0,50.2215,350.4,1199.0,893418.0,34818.941115
LongTermDebt,10816.0,11539.11469,0.0,1650.0875,4123.6085,11197.75,410157.0,26531.977254
TotalDebt,9266.0,16527.742964,0.295,2068.0,5313.0,13346.62075,1187451.0,59589.393676
StockholdersEquity,11382.0,14178.970365,-15147.0,1754.525,4476.037,12105.7,656742.0,39150.821309
TotalAssets,11384.0,54097.749253,0.423756,6590.28475,14945.0,39381.25,4357856.0,223559.189847


## Missing Values

In [ ]:
# Se separan columnas numéricas y no numéricas
df_cont = df_with_features.select_dtypes(include='number')
df_non_numeric = df_with_features.select_dtypes(exclude='number')

# Comprobar valores infinitos antes de imputación multivariable
print(np.isinf(df_cont).sum())

: 

In [ ]:
# NaN Restantes: Imputación multivariable con IterativeImputer sobre numéricas
# Imputador: Chain equations
imputer_itImp = skl_imp.IterativeImputer(max_iter=10, random_state=0)

df_cont_imputed = pd.DataFrame(imputer_itImp.fit_transform(df_cont),columns=df_cont.columns)
mostrar_missings(df_cont_imputed)

: 

In [ ]:
# Se vuelven a unir las columnas numéricas y no numéricas
df_imputed = pd.concat([df_cont_imputed, df_non_numeric], axis=1)

: 

## Transformaciones

In [ ]:
# Se calculan tamaños relativos: RelativeAssets y RelativeRevenue
df_transformed = calcular_relative_size(df_imputed)
df_transformed.info()

: 

In [ ]:
# Se expresan columnas de capitalización y de mercado en billions
cols = [
    'MarketCap', 
    'EnterpriseValue', 
    'TotalMarketAssets', 
    'TotalMarketRevenue'
    ]

for col in cols:
    df_transformed[col] = df_transformed[col] / 10**9

In [ ]:
# Convertir Sector y Industry a tipo category
df_transformed['Sector'] = df_transformed['Sector'].astype('category')
df_transformed['Industry'] = df_transformed['Industry'].astype('category')

# Valores unicos en Sector
df_transformed['Sector'].value_counts()

In [ ]:
# Valores unicos en Industry
df_transformed['Industry'].value_counts()

In [ ]:
# Distribución de variables contínuas
df_transformed.describe().round(4).T

In [ ]:
# Coeficientes de asimetría
df_transformed.select_dtypes(include="number").skew().sort_values(ascending=False)

In [ ]:
# Graficar
columna_a_graficar = 'NetDebtToEbitda' # indicar columna para el grafico
plot(df_transformed[columna_a_graficar])

In [ ]:
# Transformaciones logarítmicas

columnas_a_transformar = [ 
    'RelativeAssets',
    'RelativeRevenue'
    ]
for columna in columnas_a_transformar:
    df_transformed[columna] = df_transformed[columna].fillna(0)
    df_transformed[f'{columna}_log'] = np.log1p(df_transformed[columna])
    df_transformed.drop(columna, axis=1, inplace=True)

# Coeficientes de asimetria actualizado
df_transformed.select_dtypes(include="number").skew().sort_values(ascending=False)

In [ ]:
# Definir columnas que saltean la "winsorización"
columnas_intactas = cols_monetarias + [
    # Variables de precio (posibles label)
    'Close',
    'Open',    
    'TrailingPE',
    'EnterpriseToEbitda',
    'PriceToBook',
    # Otras
    'Date', 
    'Ticker'       
    ]

# Separar el dataset
df_passthrough = df_transformed[columnas_intactas].copy()
df_transformed_features = df_transformed.drop(columns=columnas_intactas)

## Gestión de Outliers

Se winsorizan los valores atipicos en las variables continuas que cumplan los siguientes criterios:

Para variables simetricas:
* A mas de 3 desviaciones tipicas de la media.
* Mas de 3 rangos intercuartilicos.

Para variables asimetricas (modulo del coeficiente de asimetrica mayor a 1):
* A mas de 3 MADs de la mediana.
* Mas de 3 rangos intercuartilicos.

In [ ]:
# Outliers
df_cont_transformed = df_transformed_features.select_dtypes(include="number")
df_winsor = df_cont_transformed.apply(lambda x: gestiona_outliers(x, clas='winsor'))

In [ ]:
# Coeficientes de asimetria luego de winsorizar
df_winsor.skew().sort_values(ascending=False)

In [ ]:
# Visualizar cambios
columna_a_graficar = 'RelativeAssets_log' # indicar columna para el grafico
plot(df_winsor[columna_a_graficar])

In [ ]:
df_winsor.describe().T

## Concatenación final de columnas

In [ ]:
df_non_numeric_transformed = df_transformed_features.select_dtypes(exclude='number')
# Se unen variables contínuas transformadas y variables no numéricas
df_combined = pd.concat([df_non_numeric_transformed, df_winsor], axis=1)

# Unir con las columnas que fueron salteadas
df_final = pd.concat([df_passthrough, df_combined], axis=1)
df_final.info()

In [ ]:
# Guardar datos extraidos en fichero clean_data
df_final.to_parquet(f"{data_folder}/clean_data.parquet")